# 08 · Representing Data & Engineering Features

> "Applied machine learning is basically feature engineering." The model matters less than
> how you present the data to it. This notebook covers the transformations that most often
> move the needle:

- **Categorical encoding** — turning text categories into numbers (one-hot).
- **Binning & interactions** — helping linear models capture non-linearity.
- **Automatic feature selection** — dropping features that don't help.
- The golden rule: **fit transforms on training data only**.

In [1]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt

## 1. Categorical features: one-hot encoding

Models need numbers, but "region = EMEA/APAC/AMER" is text. **One-hot encoding** creates one
0/1 column per category. Don't just map categories to 1/2/3 — that invents a fake ordering the
model will wrongly exploit.

In [2]:
df = pd.DataFrame({
    "region": ["EMEA", "APAC", "AMER", "APAC", "EMEA"],
    "plan":   ["free", "pro", "pro", "free", "enterprise"],
    "usage":  [12, 88, 45, 7, 130],
})
encoded = pd.get_dummies(df, columns=["region", "plan"], dtype=int)
print("original columns:", list(df.columns))
print("after one-hot:   ", list(encoded.columns))
encoded

original columns: ['region', 'plan', 'usage']
after one-hot:    ['usage', 'region_AMER', 'region_APAC', 'region_EMEA', 'plan_enterprise', 'plan_free', 'plan_pro']


,usage,region_AMER,region_APAC,region_EMEA,plan_enterprise,plan_free,plan_pro
0,12,0,0,1,0,1,0
1,88,0,1,0,0,0,1
2,45,1,0,0,0,0,1
3,7,0,1,0,0,1,0
4,130,0,0,1,1,0,0


## 2. Binning and interactions help linear models

A linear model sees each feature as a straight-line effect. Two tricks add flexibility without
switching models:

- **Binning** chops a continuous feature into ranges (e.g. age → child/adult/senior), letting
  each range have its own effect.
- **Interaction/polynomial features** multiply features together, capturing "the effect of A
  depends on B".

In [3]:
from sklearn.preprocessing import KBinsDiscretizer, PolynomialFeatures

x = np.linspace(0, 10, 8).reshape(-1, 1)
binner = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")
print("values:", x.ravel())
print("bins:  ", binner.fit_transform(x).ravel().astype(int))

pair = np.array([[2.0, 3.0], [4.0, 5.0]])
poly = PolynomialFeatures(degree=2, include_bias=False)
print("\noriginal pair:\n", pair)
print("with interactions [a, b, a², a·b, b²]:\n", poly.fit_transform(pair))

values: [ 0.          1.42857143  2.85714286  4.28571429  5.71428571  7.14285714
  8.57142857 10.        ]
bins:   [0 0 0 1 1 2 2 2]

original pair:
 [[2. 3.]
 [4. 5.]]
with interactions [a, b, a², a·b, b²]:
 [[ 2.  3.  4.  6.  9.]
 [ 4.  5. 16. 20. 25.]]


## 3. Automatic feature selection

More features isn't always better — noise features hurt. Selection keeps the informative ones,
which speeds training and can improve accuracy. We add pure-noise columns to a real dataset and
watch selection find the real ones.

In [4]:
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, data.target
rng = np.random.RandomState(0)
X_noisy = np.hstack([X, rng.normal(size=(X.shape[0], 30))])   # 30 real + 30 fake features
print("total features now:", X_noisy.shape[1])

base = cross_val_score(LogisticRegression(max_iter=5000), X_noisy, y, cv=5).mean()

from sklearn.pipeline import make_pipeline
sel = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=5000))
selected = cross_val_score(sel, X_noisy, y, cv=5).mean()

print(f"accuracy with all 60 features:      {base:.1%}")
print(f"accuracy after selecting best 20:   {selected:.1%}")

total features now: 60


accuracy with all 60 features:      94.4%
accuracy after selecting best 20:   94.9%


In [5]:
# Which features got selected? Almost all the real ones, few of the noise ones.
kb = SelectKBest(f_classif, k=20).fit(X_noisy, y)
chosen = np.where(kb.get_support())[0]
real_chosen = np.sum(chosen < 30)
print(f"of 20 selected: {real_chosen} are real features, {20 - real_chosen} are noise")

of 20 selected: 20 are real features, 0 are noise


## 4. The golden rule: no peeking at test data

Every transform that *learns* from data — scalers, encoders, selectors, imputers — must be
`fit` on the **training set only**, then `transform` applied to test data. Fitting on the full
dataset leaks test information and inflates your scores. Notebook 10 (Pipelines) makes this
automatic and mistake-proof.

## Recap

- **One-hot encode** categories; never fake an ordinal ordering.
- **Binning** and **interaction/polynomial** features give linear models non-linear reach.
- **Feature selection** drops noise, improving speed and sometimes accuracy.
- **Fit transforms on train only** — leakage is the silent score-inflator.

**Next:** `09 — Model Evaluation`, where we learn to trust a score.